# Audio Style Workflow（Jupyter 交互版）

本 Notebook 把 `audio_style_workflow` 包的**五段式工作流**拆开，一步步讲解并运行：

```
None ──▶ scan ──▶ split_tta ──▶ analyze ──▶ parse_styles ──▶ write_tags ──▶ [TrackTask]
         (扫描   (TTA+CUE      (HTTP 上传     (JSON 提取 /      (mutagen
         目录)   ffmpeg 切分    到分析端点,    LLM 兜底)         写自定义标签,
                 为 FLAC 分轨)  支持并发)                         支持 dry-run)
```

每条语链（pipeline）步骤都是 LangChain 的 `RunnableLambda`（纯函数：状态进、状态出），
用 `|` 串接成一条 `RunnableSequence`。单文件失败只记录错误，不中断整批。

> **分析端点**：本 Notebook 使用 sglang 部署的音乐理解模型，走 **OpenAI 兼容的**
> `/v1/chat/completions` 接口，音频以 `audio_url` 传**本机路径**（服务端直接读文件）。
> 运行前需先启动 sglang 服务（见第 3.1 节），并保证音乐库与该服务**共享文件系统**。

## 0. 准备

运行依赖见 `requirements.txt`：

```bash
pip install -r requirements.txt
```

需要 **ffmpeg** 才能演示 TTA 切分和生成示例音频（Windows 可用 `winget install ffmpeg`）；
若不可用，本 Notebook 会自动降级：仍可处理 mp3/flac 直连文件，仅跳过 TTA 演示。

In [ ]:
import os, sys, shutil, subprocess, tempfile
from pathlib import Path
import requests

# 保证能从项目根目录 import 本包（在任意工作目录启动都可靠）
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from audio_style_workflow.api.base import create_client, SongAnalysisClient, SongAnalysisError
from audio_style_workflow.api.sglang_client import SglangMusicClient, DEFAULT_SGLANG_BASE
from audio_style_workflow.cue import parse_cue_text, parse_time
from audio_style_workflow.models import AudioFormat, PipelineState, TrackTask
from audio_style_workflow.scanner import scan_directory
from audio_style_workflow.splitter import TtaSplitter, resolve_ffmpeg
from audio_style_workflow.state_store import ScanStateStore
from audio_style_workflow.style_parser import (
    StyleParser, extract_styles_from_text, DEFAULT_ANALYSIS_PROMPT,
)
from audio_style_workflow.tagger import get_styles, write_styles, write_basic_tags, DEFAULT_CUSTOM_KEY
from audio_style_workflow.logging_setup import DEFAULT_LOG_DIR, setup_logging
from audio_style_workflow.workflow import (
    WorkflowConfig, run_workflow, step_scan, step_split,
    step_analyze, step_parse, step_tag, build_workflow,
)

# 日志：log/latest.log 记录全量日志，log/details/ 存放逐曲分析明细
LOG_PATHS = setup_logging(DEFAULT_LOG_DIR)
LOG_DIR = DEFAULT_LOG_DIR

def wf_config(root, **kwargs):
    """构造 WorkflowConfig 并默认带上 log_dir（产出 log/details 明细）。"""
    kwargs.setdefault("log_dir", LOG_DIR)
    return WorkflowConfig(root=root, **kwargs)

print(f"ffmpeg: {resolve_ffmpeg() or '不可用（将跳过 TTA 演示）'}")
print(f"sglang 基地址: {DEFAULT_SGLANG_BASE}")
print(f"日志: {LOG_PATHS.latest}  明细: {LOG_PATHS.details}")
print("包已就绪")

## 1. 构造示例音乐库

用 ffmpeg 合成几段正弦音作为「歌曲」，放进一个临时目录，模拟真实的音乐库。

结构：
```
music/
├── Track A.mp3      （直连 mp3）
├── Track B.mp3      （直连 mp3）
└── 专辑专辑/
    └── album.mp3    （子目录里的直连 mp3）
```

In [ ]:
# 建一个临时音乐库
MUSIC = Path(tempfile.mkdtemp(prefix="asw_music_")).resolve()
(MUSIC / "classic_album").mkdir(parents=True, exist_ok=True)
print(f"音乐库根目录: {MUSIC}")

def make_tone(path: Path, duration: float = 3, freq: int = 440) -> bool:
    """用 ffmpeg 合成一段 mp3 音频；ffmpeg 不可用时返回 False。"""
    ffmpeg = resolve_ffmpeg()
    if not ffmpeg:
        return False
    subprocess.run(
        [ffmpeg, "-y", "-hide_banner", "-loglevel", "error", "-f", "lavfi",
         "-i", f"sine=frequency={freq}:duration={duration}",
         "-c:a", "libmp3lame", "-q:a", "9", str(path)],
        check=True, capture_output=True,
    )
    return True

samples = [
    (MUSIC / "Track A.mp3", 4, 440),
    (MUSIC / "Track B.mp3", 4, 550),
    (MUSIC / "classic_album" / "Winter.mp3", 5, 330),
    (MUSIC / "classic_album" / "Spring.mp3", 5, 385),
]
for p, dur, freq in samples:
    ok = make_tone(p, dur, freq)
    if ok:
        print(f"生成: {p.name} ({p.stat().st_size} bytes)")
    else:
        print(f"[跳过] ffmpeg 不可用，无法生成 {p.name}")

## 2. 第 1 步：`scan` —— 递归扫描目录

`scan_directory(root, skip_tag_key=...)` 递归扫描：

- `mp3` / `flac` → 直接进入 `tasks`（待分析）；
- `tta` 整轨 → 必须配对到 CUE（同名优先，其次按 `FILE` 指令回退配对）进入 `albums`；
- 给了 `skip_tag_key` 时，文件里已带该自定义标签的歌会被标记 `skip_reason`。

先看关键数据结构 `PipelineState` / `TrackTask`。

In [ ]:
state = scan_directory(MUSIC)
print(f"直连音频 {len(state.tasks)} 首, TTA 专辑 {len(state.albums)} 张, 警告 {len(state.warnings)}")
print("--- 待分析任务 ---")
for t in state.tasks:
    print(f"  - [{t.fmt.value}] {t.path.name}  来源={t.source.name}  title={t.title}")

## 3. 分步执行（手动跑一遍五个阶段）

工作流的每一步都是「配置驱动的纯函数」。下面**不调用整个管道**，而是每个阶段分开调，
直观看到状态 `PipelineState` 如何在各阶段被持续富化。

### 3.1 分析端点：sglang（OpenAI 兼容 `/v1/chat/completions`）

先启动 sglang 部署的音乐理解模型（如 moss-music-model，服务端直接读音频路径）：

```bash
sglang serve --model-path /path/to/moss-music-model --trust-remote-code
```

然后用包内置的 `SglangMusicClient(api_mode="chat", thinking_budget=50)` 创建客户端：
它对每首歌发起与你手写示例**等价**的请求——`model=default`，
`messages[].content` 里同时带 `audio_url`（解析为绝对路径）与分析文本，
并附带 `separate_reasoning=True`、`custom_logit_processor` 和
`custom_params`（含 `thinking_budget=50`）以控制思考 token 预算。

服务地址 `SGLANG_BASE` 支持三种指定方式（优先级从高到低）：

1. **命令行参数** `--sglang-base <URL>`（下个 cell 的 `argparse` 解析）；
2. **环境变量** `SGLANG_BASE`；
3. 默认值 `http://localhost:30000`。

命令行示例：

```bash
# 直接执行 notebook 并指定远端 sglang 地址
jupyter nbconvert --to notebook --execute audio_style_workflow.ipynb \
    -- --sglang-base http://192.168.1.10:30000
```
> 跨机部署（客户端 Windows / 服务端 Linux）时，服务端读不到本地路径，二选一：
> 用 `--sglang-path-map 本地前缀=服务端前缀`，或直接 `audio_base64=True` 把音频
> 以 `data:<mime>;base64,...` 内联发送（sglang 原生支持 `data:` URL）。
> 注意：部分 sglang 版本丢 `audio_url`（`prompt_tokens` 仅 ~105）。因此该客户端
> 固定走 `/generate`（`audio_data`），CLI 也不再提供 `--sglang-api`。
> 音频可用 `--sglang-audio-base64` 内联，无需共享文件系统。


In [ ]:
import argparse

# 支持用命令行参数指定 sglang 地址（优先），其次环境变量 SGLANG_BASE，最后默认 http://localhost:30000：
#   jupyter nbconvert --to notebook --execute audio_style_workflow.ipynb -- --sglang-base http://192.168.1.10:30000
# 直接在 Jupyter 里运行时，parse_known_args 会忽略内核自带的参数（如 -f kernel.json）。
_argp = argparse.ArgumentParser(add_help=False)
_argp.add_argument("--sglang-base", "--sglang_base", dest="sglang_base",
                   default=os.environ.get("SGLANG_BASE", DEFAULT_SGLANG_BASE),
                   help="sglang 服务基地址（默认读环境变量 SGLANG_BASE，再回退 http://localhost:30000）")
_args, _ = _argp.parse_known_args()
SGLANG_BASE = _args.sglang_base.rstrip("/")

_kwargs = dict(endpoint=SGLANG_BASE, api_mode="generate", model_name="default",
               max_new_tokens=1024, temperature=0.0)
# thinking budget 需要 sglang 的 custom_logit_processor 序列化串（to_str() 输出，
# 内含 dill 数据，必须由与服务端一致的 sglang 生成）；拿不到就不启用，
# 绝不把裸类名发给服务端（会让 sglang scheduler 崩溃）。
try:
    client = SglangMusicClient(thinking_budget=50, **_kwargs)
    print("thinking budget: 已启用（处理器序列化串就绪）")
except ValueError as exc:
    print("[提示] 未获取到 thinking budget 处理器串，改为不启用 budget：")
    print("      ", str(exc).splitlines()[0])
    client = SglangMusicClient(thinking_budget=None, **_kwargs)
parser = StyleParser()                 # 不注入 LLM -> 仅确定性解析
config = wf_config(MUSIC)
print(f"客户端: {client.name}  端点: {client._url_chat}")
print(f"解析器 LLM 兜底: {parser.llm_enabled}")

### 3.2 手写一次 `/generate` 请求（等价于上面的 `SglangMusicClient`）

为透明起见，这里直接用 `requests` 调用 `/generate`，观察请求体与响应结构。
响应里的 `choices[0].message.content` 是最终分析结果，
`meta_info.prompt_tokens` 能验证音频是否真的送进了模型（几百→几千为正常）。


In [ ]:
audio_file = MUSIC / "Track A.mp3"

# 把音频内联成 data URL：服务端无需与本地共享文件系统
import base64
_mime = {".mp3": "audio/mpeg", ".flac": "audio/flac", ".wav": "audio/wav"}
_m = _mime.get(audio_file.suffix.lower(), "application/octet-stream")
audio_data = f"data:{_m};base64," + base64.b64encode(audio_file.read_bytes()).decode()

sampling = {"max_new_tokens": 1024, "temperature": 0.0}
body = {
    "text": DEFAULT_ANALYSIS_PROMPT,
    "audio_data": audio_data,
    "sampling_params": sampling,
}
# 只有拿到合法的 processor 序列化串时才附带 thinking budget
if client.thinking_budget is not None and client.custom_logit_processor:
    sampling["custom_params"] = {"thinking_budget": client.thinking_budget}
    body["custom_logit_processor"] = client.custom_logit_processor

resp = requests.post(f"{SGLANG_BASE}/generate", json=body, timeout=600)
resp.raise_for_status()
data = resp.json()
print("text      :", data.get("text"))
print("prompt_tokens:", (data.get("meta_info") or {}).get("prompt_tokens"))

### 3.3 `scan` → `analyze` → `parse` → `tag`

调用各 `step_*` 纯函数，观察 `TrackTask` 从「只有路径」逐渐被填满 `analysis / styles / tagged`。
（`step_split` 暂不传 `splitter`，本目录没有 TTA，跳过即可。）

In [ ]:
s1 = step_scan(None, config)                       # scan
s2 = step_split(s1, config, splitter=None)         # split（无 TTA，原样返回）
s3 = step_analyze(s2, config, client)              # analyze（并发提交分析端点）
s4 = step_parse(s3, config, parser)                # parse_styles（提取风格）
s5 = step_tag(s4, config)                          # write_tags（写回自定义标签）

print(f"{'文件':<14}{'analysis':<10}{'styles':<20}{'BPM':<6}{'Tempo':<12}{'Key':<12}tagged")
for t in s5.tasks:
    print(f"{t.path.name:<14}{'OK' if t.analysis else '-':<10}"
          f"{','.join(t.styles):<20}{str(t.bpm or '-'):<6}"
          f"{str(t.tempo or '-'):<12}{str(t.key or '-'):<12}{t.tagged}")

In [ ]:
# 验证标签真的写进了音频文件
for p, *_ in [(MUSIC / "Track A.mp3"), (MUSIC / "classic_album" / "Winter.mp3")]:
    if p.exists():
        print(f"{p.name}: STYLE = {get_styles(p, key=DEFAULT_CUSTOM_KEY)!r}")

## 4. 端到端：LangChain `RunnableSequence`

上面分步调用等价于 `build_workflow(...)` 组装出的那条 `RunnableSequence`。
`build_workflow` 用 `RunnableLambda | RunnableLambda | ...` 串接五步，
`.invoke(None)` 一次跑完整条链，返回最终的 `PipelineState`。

In [ ]:
workflow = build_workflow(client, StyleParser(), wf_config(MUSIC))
print("管道步骤:", " -> ".join(step.name for step in workflow.steps))
print("数据类型:", type(workflow).__name__)

final = workflow.invoke(None)   # 输入 None，得到 PipelineState
print(f"\n返回 {len(final.tasks)} 个 TrackTask；带风格: "
      f"{sum(1 for t in final.tasks if t.styles)}")

## 5. 便捷入口 `run_workflow` + 汇总报告

`run_workflow(...)` 封装了：自动构造 `splitter`（ffmpeg 缺失优雅降级）、
维护 `skip-scanned` 扫描记录（`reset_state` / `load` / 运行时 `mark` / 结束时 `save`）、
执行 LCEL 管道并生成 `RunReport`。

`RunReport` 提供 `analyzed / parsed_by_json / tagged / skipped / failed` 等统计属性。

In [ ]:
report = run_workflow(
    client=client,          # 同一个 sglang /v1/chat/completions 客户端
    style_parser=StyleParser(),
    config=wf_config(MUSIC),
)
report.print_summary()

## 6. dry-run：演练但不落盘

`dry_run=True` 时只**解析并展示**将写入的标签（`would_write`），不写任何文件，
也不产生扫描记录——保持预览无副作用。

In [ ]:
cfg_dry = wf_config(MUSIC, dry_run=True, skip_tagged=True)
# 全部歌曲此时都已带标签，看 dry-run 仍然会解析展示
report_dry = run_workflow(
    client=client,
    style_parser=StyleParser(),
    config=cfg_dry,
)
report_dry.print_summary()
print("\n--- 将写入的值 (would_write) ---")
for t in report_dry.tasks:
    print(f"  {t.path.name:<26} {t.would_write}")

## 7. 断点续跑：`skip-scanned`

`skip_scanned=True` 时用独立 JSON 状态文件（默认 `<root>/.audio_style_scan.json`）记录每个文件的
处理指纹（`大小 + mtime`）。首次运行全量处理并生成记录；再次运行未变化的歌直接跳过，
只处理新增/变动的文件。文件被改动后指纹变化会自动重新分析。

In [ ]:
# 换一个新目录，跑两次 skip-scanned
MUSIC2 = Path(tempfile.mkdtemp(prefix="asw_music2_")).resolve()
for name, f in [("One.mp3", 440), ("Two.mp3", 520)]:
    make_tone(MUSIC2 / name, 3, f)
sglang_chat = lambda: client
print("首次运行（全量）:")
r1 = run_workflow(sglang_chat(), StyleParser(),
                  wf_config(MUSIC2, skip_scanned=True))
r1.print_summary()

print("\n再次运行（应全部命中扫描记录跳过）:")
r2 = run_workflow(sglang_chat(), StyleParser(),
                  wf_config(MUSIC2, skip_scanned=True))
r2.print_summary()

state_file = r2.state_path
print(f"\n状态文件: {state_file} (存在={state_file.exists() if state_file else False})")

## 8. TTA 整轨切分（可选，需要 ffmpeg + 真实整轨）

TTA 与 FLAC 同为无损编码，`tta -> flac` 逐轨转码无质量损失。切点取每轨 `INDEX 01`
（缺失回退第一个 INDEX），帧时间按 75fps 换算，使用输出级精确 seek + `.part` 临时文件。

下面先演示 **CUE 解析**（无任何外部依赖），再尝试真实整轨切分（ffmpeg 可用时）。

In [ ]:
# 8.1 解析一段 CUE 文本
cue_text = '''\
REM GENRE 古典
TITLE \"The Piano Works\"
PERFORMER \"Famous Artist\"
FILE \"album.tta\" WAVE
  TRACK 01 AUDIO
    TITLE \"Nocturne\"
    INDEX 01 00:00:00
  TRACK 02 AUDIO
    TITLE \"Waltz\"
    INDEX 01 03:45:20
'''
sheet = parse_cue_text(cue_text)
print(f"专辑: {sheet.title}  /  {sheet.performer}")
print(f"FILE 指令: {sheet.file_name}")
for t in sheet.tracks:
    print(f"  轨 {t.number}: {t.title!r} @ {t.start_sec:.2f}s  ({t.start_sec})")

In [ ]:
# 8.2 若 ffmpeg 支持 tta 编码，合成一张整轨 .tta + .cue 并切分（否则跳过）
ffmpeg = resolve_ffmpeg()
album_dir = MUSIC / "whole_album"
try:
    album_dir.mkdir(exist_ok=True)
    tta_path = album_dir / "album.tta"
    subprocess.run(
        [ffmpeg, "-y", "-hide_banner", "-loglevel", "error", "-f", "lavfi",
         "-i", "sine=frequency=400:duration=8", "-c:a", "tta", str(tta_path)],
        check=True, capture_output=True,
    )
    (album_dir / "album.cue").write_text(cue_text, encoding="utf-8")
    print(f"生成整轨: {tta_path}")
    tta_ok = True
except Exception as exc:  # tta 编码不可用等情况
    tta_ok = False
    print(f"[跳过] TTA 整轨切分演示不可用: {exc}")

if tta_ok:
    splitter = TtaSplitter()
    scanned = scan_directory(MUSIC)
    album = next(a for a in scanned.albums if a.tta == tta_path)
    tracks = splitter.split(album, out_dir=album_dir / "split")
    print(f"\n切出 {len(tracks)} 个分轨:")
    for tr in tracks:
        print(f"  - {tr.path.name}  title={tr.title!r} artist={tr.artist!r}  fmt={tr.fmt.value}")

## 9. LLM 兜底解析（解析器演示，非主流程）

本 Notebook 的主流程走 sglang 的 OpenAI 兼容接口，风格在 `SglangMusicClient` 内
已用 `extract_styles_from_text` 从模型输出首行确定性解析。
本节换一种视角：当分析端点返回**非结构化文本**（如自然语言描述、没有可识别的风格键）时，
`StyleParser` 的确定性提取会失败。此时可注入任意的 LangChain ChatModel 做结构化提取兜底：

1. 用进程内 `mock` 客户端切到 `mode="freeform"` 来**模拟**一个返回非结构化响应的端点；
2. 注入 `ChatOpenAI(model=...)`（需 `langchain-openai` + `OPENAI_API_KEY`）让解析器兜底。

下面先看**不注入 LLM 时会抛什么错**。

In [ ]:
from audio_style_workflow.style_parser import StyleParseError

freeform = create_client("mock", mode="freeform")   # 返回非结构化响应
f = freeform.analyze(MUSIC / "Track A.mp3")
print("自由文本响应:", f)

print("\n不注入 LLM 时的确定性解析结果:")
try:
    StyleParser().parse(f, source_name="Track A.mp3")
except StyleParseError as exc:
    print("  !!", exc)

> 若已安装 `langchain-openai` 并设置 `OPENAI_API_KEY`，可取消下面一行注释，走 LLM 兜底：
>
> ```python
> from langchain_openai import ChatOpenAI
> llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
> styles, source, conf = StyleParser(llm).parse(f, source_name="Track A.mp3")
> print(styles, source)
> ```

另外，本地生成式模型（MusicFlamingo / sglang 部署模型）输出首行风格标签时，
可用 `extract_styles_from_text` 做零成本确定性解析（同样无需 LLM）：

In [ ]:
raw_output = "Pop, R&B, Jazz\n这是一首融合爵士与都市律动的曲子..."
print("首行提取:", extract_styles_from_text(raw_output))
print("带前缀:", extract_styles_from_text("Styles: Ambient, Electronic"))

## 10. 接入自定义分析服务

接入新服务只需继承 `SongAnalysisClient` 并实现 `analyze` 返回约定的响应 dict，
然后把它传给 `run_workflow(client=...)`，工作流其余部分完全不变。

In [ ]:
class MyServiceClient(SongAnalysisClient):
    name = "my-service"
    def analyze(self, audio_path):
        # 你的传输方式：预签名 URL / gRPC / 消息队列……
        return {"status": "ok", "styles": ["City Pop", "Disco"], "confidence": 0.9}

MUSIC3 = Path(tempfile.mkdtemp(prefix="asw_music3_")).resolve()
make_tone(MUSIC3 / "Neon.mp3", 3, 600)

r = run_workflow(
    client=MyServiceClient(),
    style_parser=StyleParser(),
    config=wf_config(MUSIC3),
)
r.print_summary()
t = r.tasks[0]
print(f"\n{t.path.name} -> styles={t.styles} 标签写回={get_styles(t.path)!r}")

## 11. 命令行方式（对应 `main.py`）

以上 API 调用等价于 CLI 入口 `python -m audio_style_workflow.main`。
sglang 固定走 `/generate`（不提供 `--sglang-api`），音频可用 base64 内联：

```bash
# 1) 默认地址 http://localhost:30000，base64 内联音频
python -m audio_style_workflow.main "D:\我的音乐" \
    --sglang --sglang-audio-base64 \
    --max-new-tokens 1024 --skip-scanned

# 2) 指定服务地址
python -m audio_style_workflow.main "D:\我的音乐" \
    --sglang http://192.168.1.10:30000 --sglang-audio-base64 \
    --custom-tag STYLE --skip-scanned

# 3) dry-run 演练（不写标签、不产生扫描记录）
python -m audio_style_workflow.main "./music" --sglang --sglang-audio-base64 --dry-run
```


In [ ]:
!python -m audio_style_workflow.main "{MUSIC}" --sglang "{SGLANG_BASE}" --sglang-audio-base64 --skip-scanned

## 12. 日志、逐曲明细与歌曲评价

工作流的日志由 `logging_setup` 模块管理，默认写到 `--log-dir`（`log/`）：

```
log/
├── latest.log              # 全量日志（每次运行覆盖）
├── details/                # 每首歌的原始输入输出 JSON
│   └── <歌名>__<hash>.json
└── Genre description/      # 每首歌的完整评价（可读文本）
    └── <歌名>__<hash>.txt
```

- `details/*.json`：原始**输入**（音频路径/大小、客户端、端点、模式、提示词）
  与原始**输出**（分析端点返回的完整响应，sglang 还含 `raw_response`）；
- `Genre description/*.txt`：该曲的完整评价（模型 `description`，
  thinking 未闭合时取自 `reasoning_content`），文件头会标注风格与来源；
  文件名与 `details` 同名前缀，可直接对照。

`--log-dir`（CLI）或 `WorkflowConfig(log_dir=...)`（库/Notebook）可指定目录；
本 Notebook 通过 `wf_config` 默认写入 `log/`。

In [ ]:
import json

print("=== latest.log（末尾 600 字）===")
print(LOG_PATHS.latest.read_text(encoding="utf-8")[-600:])

detail_files = sorted(LOG_PATHS.details.glob("*.json"))
print(f"\n=== details/ 共 {len(detail_files)} 份，示例 ===")
if detail_files:
    data = json.loads(detail_files[0].read_text(encoding="utf-8"))
    print(detail_files[0].name)
    print(json.dumps(data, ensure_ascii=False, indent=2)[:900])

desc_files = sorted(LOG_PATHS.descriptions.glob("*.txt"))
print(f"\n=== Genre description/ 共 {len(desc_files)} 份，示例 ===")
if desc_files:
    print(desc_files[0].name)
    print("-" * 50)
    print(desc_files[0].read_text(encoding="utf-8")[:900])

## 小结

到这里你已经把 `audio_style_workflow` 五段式工作流完整跑了一遍：

| 阶段 | 函数 | 输入 → 输出 |
|---|---|---|
| 扫描 | `scan_directory` / `step_scan` | 目录 → `PipelineState`(tasks+albums) |
| 切分 | `TtaSplitter.split` / `step_split` | TTA+CUE → FLAC 分轨并入 tasks |
| 分析 | `SglangMusicClient(api_mode="chat", …).analyze` / `step_analyze` | 音频 → 原始响应 dict |
| 解析 | `StyleParser.parse` + `extract_musical_meta` / `step_parse` | 响应 → `styles` + `bpm/tempo/key` |
| 写标签 | `write_styles` + `write_extra_tags` / `step_tag` | styles、BPM/Tempo/Key → mutagen 标签 |

- **关键设计**：每步是 LangChain `RunnableLambda` 纯函数，用 `|` 串成 `RunnableSequence`；
  单文件失败只记录错误，不中断整批。
- **断点续跑**：`skip-tagged`（文件内标签）与 `skip-scanned`（独立状态文件指纹）两种机制。
- **接入新服务**：继承 `SongAnalysisClient` 即可，工作流其余部分不变。